In [ ]:
import sys
print(sys.executable)


In [ ]:
import challenge_ml
print(challenge_ml.__file__)


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

from challenge_ml.feature import build_training_data
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



## Split

In [ ]:
# Split aléatoire :

x_train, y_train, x_test, test_ids = build_training_data()

X_train_rand, X_valid_rand, y_train_rand, y_valid_rand = train_test_split(
    x_train,
    y_train,
    test_size=0.2,
    random_state=42,
    shuffle=True,
  )


In [ ]:
# Split orienté Humidity :
humidity = x_train["Humidity"]

humidity_threshold = humidity.quantile(0.8)
humidity_shift_mask = humidity >= humidity_threshold

X_train_shift = x_train.loc[~humidity_shift_mask].copy()
X_valid_shift = x_train.loc[humidity_shift_mask].copy()
y_train_shift = y_train.loc[~humidity_shift_mask].copy()
y_valid_shift = y_train.loc[humidity_shift_mask].copy()

print("Humidity shift split")
print("threshold:", round(humidity_threshold, 4))
print("X_train:", X_train_shift.shape, "X_valid:", X_valid_shift.shape)
print("y_train:", y_train_shift.shape, "y_valid:", y_valid_shift.shape)

humidity_train_shift = humidity.loc[X_train_shift.index]
humidity_valid_shift = humidity.loc[X_valid_shift.index]

print("Humidity mean train:", round(humidity_train_shift.mean(), 4))
print("Humidity mean valid:", round(humidity_valid_shift.mean(), 4))
print("Humidity median train:", round(humidity_train_shift.median(), 4))
print("Humidity median valid:", round(humidity_valid_shift.median(), 4))


## Train

In [ ]:
def evaluate_model(y_true, y_pred, name):
    y_pred = np.clip(y_pred, 0, 1)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred, multioutput="uniform_average")

    print(name)
    print("MAE :", round(mae, 5))
    print("RMSE:", round(rmse, 5))
    print("R2  :", round(r2, 5))




In [ ]:
# modèle random

xgb_random = MultiOutputRegressor(
    XGBRegressor(
        objective="reg:squarederror",
        n_estimators=100,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )
)

xgb_random.fit(X_train_rand, y_train_rand)

y_pred_rand = xgb_random.predict(X_valid_rand)

evaluate_model(y_valid_rand, y_pred_rand, "XGBoost - random split")



In [ ]:
xgb_shift = MultiOutputRegressor(
    XGBRegressor(
        objective="reg:squarederror",
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )
)

xgb_shift.fit(X_train_shift, y_train_shift)

y_pred_shift = xgb_shift.predict(X_valid_shift)

evaluate_model(y_valid_shift, y_pred_shift, "XGBoost - shift sur humidity")


## Submit 

In [ ]:
x_train, y_train, x_test, test_ids = build_training_data()

xgb_submission = MultiOutputRegressor(
    XGBRegressor(
        objective="reg:squarederror",
        n_estimators=100,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )
)

xgb_submission.fit(x_train, y_train)

y_test_pred = xgb_submission.predict(x_test)
y_test_pred = np.clip(y_test_pred, 0, 1)

submission = pd.DataFrame(
    y_test_pred,
    columns=y_train.columns,
)

submission.insert(0, "ID", test_ids)
submission["c15"] = 0.0

submission = submission[
    [
        "ID",
        "c01", "c02", "c03", "c04", "c05", "c06", "c07", "c08", "c09", "c10",
        "c11", "c12", "c13", "c14", "c15", "c16", "c17", "c18", "c19", "c20",
        "c21", "c22", "c23",
    ]
]


submission.to_csv("../data/processed/submission_xgboost_random.csv", index=False)




In [ ]:
submission.head()
